# **FOREWORD**

This kernel is a replica of the topper solution at MCAA 2023. I have made suitable changes to the code to improve the readibility.

### **VERSION DETAILS**
Here, I am clipping predictions between a low and high bound - values to be decided using low and high variables

# **IMPORTS**

In [1]:
!uv pip install -q scikit-learn==1.7.2 xgboost==3.2.0 --system

In [2]:
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import *
from scipy.interpolate import UnivariateSpline
import statsmodels.api as sm
import matplotlib.pyplot as plt
import collections
from colorama import Fore, Back, Style
from warnings import filterwarnings
filterwarnings("ignore")

def PrintColor(text: str, color = Fore.BLUE, style = Style.BRIGHT) :
    print(color + style + text + Style.RESET_ALL)

pd.set_option("display.max_column", 999)

In [3]:
DATA_PATH = '/kaggle/input/competitions/march-machine-learning-mania-2026/'
test_req  = False

if test_req :
    repeat_cv = 2
else:
    repeat_cv = 10

low  = 0.02
high = 1 - low

# **PREPROCESSING**

In [4]:
def prepare_data(df):
    dfswap = df[['Season', 'DayNum', 'LTeamID', 'LScore', 'WTeamID', 'WScore', 'WLoc', 'NumOT', 
    'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF', 
    'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF']]

    dfswap.loc[df['WLoc'] == 'H', 'WLoc'] = 'A'
    dfswap.loc[df['WLoc'] == 'A', 'WLoc'] = 'H'
    df.columns.values[6] = 'location'
    dfswap.columns.values[6] = 'location'    
      
    df.columns = [x.replace('W','T1_').replace('L','T2_') for x in list(df.columns)]
    dfswap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(dfswap.columns)]

    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output.loc[output.location=='N','location'] = '0'
    output.loc[output.location=='H','location'] = '1'
    output.loc[output.location=='A','location'] = '-1'
    output.location = output.location.astype(int)
    
    output['PointDiff'] = output['T1_Score'] - output['T2_Score']
    
    return output

def team_quality(season):
    formula = 'win~-1+T1_TeamID+T2_TeamID'
    data = regular_season_effects.loc[regular_season_effects.Season == season, :]
    if data['T1_TeamID'].nunique() < 2 or data['T2_TeamID'].nunique() < 2:
        return pd.DataFrame(columns=['TeamID', 'quality', 'Season'])
    glm = sm.GLM.from_formula(formula=formula, data=data, family=sm.families.Binomial()).fit()
    
    quality = pd.DataFrame(glm.params).reset_index()
    quality.columns = ['TeamID', 'quality']
    quality['Season'] = season
    quality = quality.loc[quality.TeamID.str.contains('T1_')].reset_index(drop=True)
    quality['TeamID'] = quality['TeamID'].apply(lambda x: x[10:14]).astype(int)
    return quality

In [5]:

tourney_results = pd.concat([
    pd.read_csv(DATA_PATH + "MNCAATourneyDetailedResults.csv"),
    pd.read_csv(DATA_PATH + "WNCAATourneyDetailedResults.csv"),
], ignore_index=True)

seeds = pd.concat([
    pd.read_csv(DATA_PATH + "MNCAATourneySeeds.csv"),
    pd.read_csv(DATA_PATH + "WNCAATourneySeeds.csv"),
], ignore_index=True)

regular_results = pd.concat([
    pd.read_csv(DATA_PATH + "MRegularSeasonDetailedResults.csv"),
    pd.read_csv(DATA_PATH + "WRegularSeasonDetailedResults.csv"),
], ignore_index=True)

PrintColor(f"---> Preparing results swap data")

regular_results_swap = \
regular_results[[
    'Season', 'DayNum', 'LTeamID', 'LScore', 'WTeamID', 'WScore', 'WLoc', 'NumOT', 
    'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF', 
    'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF']
]

regular_results_swap.loc[regular_results['WLoc'] == 'H', 'WLoc'] = 'A'
regular_results_swap.loc[regular_results['WLoc'] == 'A', 'WLoc'] = 'H'
regular_results.columns.values[6] = 'location'
regular_results_swap.columns.values[6] = 'location'

regular_results.columns      = [x.replace('W','T1_').replace('L','T2_') for x in list(regular_results.columns)]
regular_results_swap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(regular_results.columns)]

regular_data = pd.concat([regular_results, regular_results_swap]).sort_index().reset_index(drop = True)

PrintColor(f"---> Preparing regular results data")

tourney_results = pd.concat([
    pd.read_csv(DATA_PATH + "MNCAATourneyDetailedResults.csv"),
    pd.read_csv(DATA_PATH + "WNCAATourneyDetailedResults.csv"),
], ignore_index=True)

seeds = pd.concat([
    pd.read_csv(DATA_PATH + "MNCAATourneySeeds.csv"),
    pd.read_csv(DATA_PATH + "WNCAATourneySeeds.csv"),
], ignore_index=True)

regular_results = pd.concat([
    pd.read_csv(DATA_PATH + "MRegularSeasonDetailedResults.csv"),
    pd.read_csv(DATA_PATH + "WRegularSeasonDetailedResults.csv"),
], ignore_index=True)

regular_data = prepare_data(regular_results)
tourney_data = prepare_data(tourney_results)

PrintColor(f"---> Preparing season statistics data")

boxscore_cols = \
['T1_Score', 'T2_Score', 
 'T1_FGM', 'T1_FGA', 'T1_FGM3', 'T1_FGA3', 'T1_FTM', 'T1_FTA', 'T1_OR', 'T1_DR', 'T1_Ast', 'T1_TO', 'T1_Stl', 'T1_Blk', 'T1_PF', 
 'T2_FGM', 'T2_FGA', 'T2_FGM3', 'T2_FGA3', 'T2_FTM', 'T2_FTA', 'T2_OR', 'T2_DR', 'T2_Ast', 'T2_TO', 'T2_Stl', 'T2_Blk', 'T2_PF', 
 'PointDiff'
]

boxscore_cols = \
[
    'T1_FGM', 'T1_FGA', 'T1_FGM3', 'T1_FGA3', 'T1_OR', 'T1_Ast', 'T1_TO', 'T1_Stl', 'T1_PF', 
    'T2_FGM', 'T2_FGA', 'T2_FGM3', 'T2_FGA3', 'T2_OR', 'T2_Ast', 'T2_TO', 'T2_Stl', 'T2_Blk',  
    'PointDiff'
]

season_statistics = regular_data.groupby(["Season", 'T1_TeamID'])[boxscore_cols].agg("mean").reset_index()
season_statistics.columns = [''.join(col).strip() for col in season_statistics.columns.values]
season_statistics_T1 = season_statistics.copy()
season_statistics_T2 = season_statistics.copy()
season_statistics_T1.columns = ["T1_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T1.columns)]
season_statistics_T2.columns = ["T2_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T2.columns)]
season_statistics_T1.columns.values[0] = "Season"
season_statistics_T2.columns.values[0] = "Season"

tourney_data = tourney_data[['Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID' ,'T2_Score']]
tourney_data = pd.merge(tourney_data, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
tourney_data = pd.merge(tourney_data, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')

PrintColor(f"---> Preparing last 14-day results data")

last14days_stats_T1 = regular_data.loc[regular_data.DayNum>118].reset_index(drop=True)
last14days_stats_T1['win'] = np.where(last14days_stats_T1['PointDiff']>0,1,0)
last14days_stats_T1 = last14days_stats_T1.groupby(['Season','T1_TeamID'])['win'].mean().reset_index(name='T1_win_ratio_14d')
last14days_stats_T2 = regular_data.loc[regular_data.DayNum>118].reset_index(drop=True)
last14days_stats_T2['win'] = np.where(last14days_stats_T2['PointDiff']<0,1,0)
last14days_stats_T2 = last14days_stats_T2.groupby(['Season','T2_TeamID'])['win'].mean().reset_index(name='T2_win_ratio_14d')

tourney_data = pd.merge(tourney_data, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
tourney_data = pd.merge(tourney_data, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')

PrintColor(f"---> Preparing regular season season effects data")

regular_season_effects = regular_data[['Season','T1_TeamID','T2_TeamID','PointDiff']].copy()
regular_season_effects['T1_TeamID'] = regular_season_effects['T1_TeamID'].astype(str)
regular_season_effects['T2_TeamID'] = regular_season_effects['T2_TeamID'].astype(str)
regular_season_effects['win'] = np.where(regular_season_effects['PointDiff']>0,1,0)
march_madness = pd.merge(seeds[['Season','TeamID']],seeds[['Season','TeamID']],on='Season')
march_madness.columns = ['Season', 'T1_TeamID', 'T2_TeamID']
march_madness.T1_TeamID = march_madness.T1_TeamID.astype(str)
march_madness.T2_TeamID = march_madness.T2_TeamID.astype(str)
regular_season_effects = pd.merge(regular_season_effects, march_madness, on = ['Season','T1_TeamID','T2_TeamID'])
regular_season_effects.shape

PrintColor(f"---> Preparing team quality data")

formula = 'win~-1+T1_TeamID+T2_TeamID'
glm = sm.GLM.from_formula(formula=formula, 
                          data=regular_season_effects.loc[regular_season_effects.Season==2010,:], 
                          family=sm.families.Binomial()).fit()

quality = pd.DataFrame(glm.params).reset_index()
glm_quality = pd.concat([team_quality(yyyy) for yyyy in range(2010, 2026, 1)]).reset_index(drop=True)

glm_quality_T1 = glm_quality.copy()
glm_quality_T2 = glm_quality.copy()
glm_quality_T1.columns = ['T1_TeamID','T1_quality','Season']
glm_quality_T2.columns = ['T2_TeamID','T2_quality','Season']

tourney_data = pd.merge(tourney_data, glm_quality_T1, on = ['Season', 'T1_TeamID'], how = 'left')
tourney_data = pd.merge(tourney_data, glm_quality_T2, on = ['Season', 'T2_TeamID'], how = 'left')

PrintColor(f"---> Preparing team seeds data")
seeds['seed']    = seeds['Seed'].apply(lambda x: int(x[1:3]))
seeds_T1         = seeds[['Season','TeamID','seed']].copy()
seeds_T2         = seeds[['Season','TeamID','seed']].copy()
seeds_T1.columns = ['Season','T1_TeamID','T1_seed']
seeds_T2.columns = ['Season','T2_TeamID','T2_seed']
tourney_data     = pd.merge(tourney_data, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
tourney_data     = pd.merge(tourney_data, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
tourney_data["Seed_diff"] = tourney_data["T1_seed"] - tourney_data["T2_seed"]

PrintColor(
    f"---> Completed feature engineering", color = Fore.RED
)

features =\
list(season_statistics_T1.columns[2:999]) + \
list(season_statistics_T2.columns[2:999]) + \
list(seeds_T1.columns[2:999]) + \
list(seeds_T2.columns[2:999]) + \
list(last14days_stats_T1.columns[2:999]) + \
list(last14days_stats_T2.columns[2:999]) + \
["Seed_diff"] + ["T1_quality","T2_quality"]

y = tourney_data['T1_Score'] - tourney_data['T2_Score']
X = tourney_data[features].values

PrintColor(
    f"---> Shapes = {X.shape} {y.shape}", color = Fore.BLACK
)

---> Preparing results swap data
---> Preparing regular results data
---> Preparing season statistics data
---> Preparing last 14-day results data
---> Preparing regular season season effects data
---> Preparing team quality data
---> Preparing team seeds data
---> Completed feature engineering
---> Shapes = (4820, 45) (4820,)


# **MODEL TRAINING**

In [6]:
def cauchyobj(preds, dtrain):
    labels = dtrain.get_label()
    c = 5000 
    x =  preds-labels    
    grad = x / (x**2/c**2+1)
    hess = -c**2*(x**2-c**2)/(x**2+c**2)**2
    return grad, hess

In [7]:
X      = tourney_data[features].values
dtrain = xgb.DMatrix(X, label = y)
param  = \
{
    'eval_metric'       :  'mae',
    'learning_rate'     :  0.02,
    'subsample'         :  0.35,
    'colsample_bytree'  :  0.7,
    'num_parallel_tree' :  10 ,
    'min_child_weight'  :  40 ,
    'gamma'             :  10 ,
    'max_depth'         :  4,
    "verbosity"         :  0,
}

xgb_cv    = []

PrintColor(
    f"---> OFFLINE MODEL TRAINING", color = Fore.RED
)

for i in range(repeat_cv): 
    PrintColor(f"Repeater {i}")
    xgb_cv.append(
        xgb.cv(
          params = param,
          dtrain = dtrain,
          obj    = cauchyobj,
          num_boost_round = 3000,
          folds = KFold(n_splits = 5, shuffle = True, random_state = i),
          early_stopping_rounds = 25,
          verbose_eval = 0
        )
    )
    
iteration_counts = [np.argmin(x['test-mae-mean'].values) for x in xgb_cv]
val_mae          = [np.min(x['test-mae-mean'].values) for x in xgb_cv]

print()
oof_preds = []
for i in range(repeat_cv):
    preds = y.copy()
    kfold = KFold(n_splits = 5, shuffle = True, random_state = i)  
    
    for train_index, val_index in kfold.split(X,y):
        dtrain_i = xgb.DMatrix(X[train_index], label = y[train_index])
        dval_i = xgb.DMatrix(X[val_index], label = y[val_index])  
        model = xgb.train(
              params = param,
              dtrain = dtrain_i,
              num_boost_round = iteration_counts[i],
              verbose_eval  = 0
        )
        preds[val_index] = model.predict(dval_i)
    oof_preds.append(np.clip(preds,-30,30))

val_cv       = []
spline_model = []

print()
for i in range(repeat_cv):
    dat = list(zip(oof_preds[i],np.where(y>0,1,0)))
    dat = sorted(dat, key = lambda x: x[0])
    datdict = {}
    for k in range(len(dat)):
        datdict[dat[k][0]]= dat[k][1]
        
    spline_model.append(UnivariateSpline(list(datdict.keys()), list(datdict.values())))
    spline_fit = spline_model[i](oof_preds[i])
    spline_fit = np.clip(spline_fit, low, high)

    val_cv.append(
        pd.DataFrame(
            {"y"     :  np.where(y > 0,1,0), 
             "pred"  :  spline_fit, 
             "season":  tourney_data.Season
            }
        )
    )

    nspace = 3 if i >=10 else 4
    PrintColor(
        f"Score{i} =  {' ' * nspace}{brier_score_loss(np.where(y > 0,1,0), spline_fit) :,.8f}"
    ) 

print()
val_cv = pd.concat(val_cv)
display(
    val_cv.
    groupby('season').
    apply(lambda x: brier_score_loss(x.y, x.pred))
)


---> OFFLINE MODEL TRAINING
Repeater 0
Repeater 1
Repeater 2
Repeater 3
Repeater 4
Repeater 5
Repeater 6
Repeater 7
Repeater 8
Repeater 9


Score0 =      0.17018485
Score1 =      0.16951220
Score2 =      0.17045341
Score3 =      0.17011888
Score4 =      0.17041805
Score5 =      0.16977277
Score6 =      0.17040870
Score7 =      0.16998081
Score8 =      0.16988353
Score9 =      0.17029749



season
2003    0.185164
2004    0.173922
2005    0.172143
2006    0.200427
2007    0.147189
2008    0.161880
2009    0.161464
2010    0.168054
2011    0.175844
2012    0.168167
2013    0.175806
2014    0.173342
2015    0.145586
2016    0.188775
2017    0.164410
2018    0.185288
2019    0.151694
2021    0.185780
2022    0.189991
2023    0.181724
2024    0.161200
2025    0.130826
dtype: float64

# **SUBMISSION**

This is the stage-1 submission file. Please revisit and change this later when stage-2 will be available

In [8]:
 
sub = pd.read_csv(DATA_PATH + "SampleSubmissionStage2.csv")
sub['Season'] = sub['ID'].apply(lambda x: int(x.split('_')[0]))
sub["T1_TeamID"] = sub['ID'].apply(lambda x: int(x.split('_')[1]))
sub["T2_TeamID"] = sub['ID'].apply(lambda x: int(x.split('_')[2]))

sub = pd.merge(sub, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
sub = pd.merge(sub, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')
sub = pd.merge(sub, glm_quality_T1, on = ['Season', 'T1_TeamID'], how = 'left')
sub = pd.merge(sub, glm_quality_T2, on = ['Season', 'T2_TeamID'], how = 'left')
sub = pd.merge(sub, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
sub = pd.merge(sub, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
sub = pd.merge(sub, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
sub = pd.merge(sub, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')
sub["Seed_diff"] = sub["T1_seed"] - sub["T2_seed"]

Xsub  = sub[features].values
dtest = xgb.DMatrix(Xsub)

sub_models = []
for i in range(repeat_cv):
    sub_models.append(
        xgb.train(
          params = param,
          dtrain = dtrain,
          num_boost_round = int(iteration_counts[i] * 1.05),
          verbose_eval = 0
        )
    )

sub_preds = []
for i in range(repeat_cv):
    sub_preds.append(
        np.clip(
            spline_model[i](np.clip(sub_models[i].predict(dtest), -30, 30 ) ),
            low, 
            high
        )
    )
    
sub["Pred"] = pd.DataFrame(sub_preds).mean(axis=0)
sub[['ID','Pred']].to_csv("submission.csv", index = None)

In [9]:
!ls
print()
!head submission.csv

if test_req : 
    PrintColor(
        f"\n\n---> THIS IS A TEST VERSION", 
        color = Fore.RED
    )
else:
    pass

__notebook__.ipynb  submission.csv

ID,Pred
2026_1101_1102,0.21698159434781467
2026_1101_1103,0.06133637522054033
2026_1101_1104,0.11781902899843774
2026_1101_1105,0.19146057660329857
2026_1101_1106,0.1767965769105268
2026_1101_1107,0.1409270995049847
2026_1101_1108,0.2261636367494296
2026_1101_1110,0.15959820725189855
2026_1101_1111,0.1226036776660194
